**Bayes Algorithm**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Paste your directory path in the 'dir' variable**

In [2]:
import os
dir = '/content/drive/MyDrive/Colab Notebooks/Fake Review Detection'
os.chdir(dir)

**Paste the directory path of preprocessed data in the variable 'csv_path'**

In [4]:
import pandas as pd

# Replace 'path/to/' with the actual directory if needed.
# If the file is in the same directory as your notebook, just use the filename.
csv_path = './Preprocessed Data/fake reviews dataset_final_text_preprocessed_v3.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path)

# Check the first few rows to ensure data is loaded correctly
print(df.head())
print(df.info())


   Unnamed: 0  label                                               text
0           0      1  good canned asparagus used get canned asparagu...
1           1      1  didnt buy particular one bought bigger one sal...
2           2      0  supposedly great chew toy problem kind hard pu...
3           3      1  wanted try new coffee company found brooklyn c...
4           4      1  dont like ginger youre going like gold kilis g...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 884501 entries, 0 to 884500
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  884501 non-null  int64 
 1   label       884501 non-null  int64 
 2   text        884471 non-null  object
dtypes: int64(2), object(1)
memory usage: 20.2+ MB
None


In [ ]:
print(df['text'].shape)
print(df['label'].shape)

(884501,)
(884501,)


In [5]:
from sklearn.model_selection import train_test_split

# First, split off the training set (70%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

# Next, split the remaining 30% into validation (20%) and test (10%)
# Since 20% is two-thirds of the remaining 30% and 10% is one-third,
# we use test_size=1/3 for this second split.
val_df, test_df = train_test_split(temp_df, test_size=1/3, random_state=42, stratify=temp_df['label'])

# Check the sizes
print("Training set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))


Training set size: 619150
Validation set size: 176900
Test set size: 88451


In [6]:
from transformers import BertTokenizer
# Initialize the tokenizer for BERT-base-uncased (you can change this to another model if needed)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [7]:
# Define a function to tokenize a dataframe's text column
def tokenize_texts(texts, max_length=128):
    # The tokenizer returns a dictionary with input_ids, attention_mask, and token_type_ids (if applicable)
    encoded = tokenizer(
        texts.tolist(),            # Convert the pandas Series to a list
        add_special_tokens=True,   # Add [CLS] and [SEP] tokens
        max_length=max_length,     # Pad or truncate to this length
        padding='max_length',      # Pad all sequences to max_length
        truncation=True,           # Truncate sequences longer than max_length
        return_attention_mask=True, # Return attention masks
        return_tensors='pt'        # Return PyTorch tensors
    )
    return encoded

# Optionally, convert the text column to strings if they aren't already
train_df['text'] = train_df['text'].astype(str)
val_df['text'] = val_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)


# Tokenize each split
train_encodings = tokenize_texts(train_df['text'])
val_encodings = tokenize_texts(val_df['text'])
test_encodings = tokenize_texts(test_df['text'])

# Extract labels as tensors
import torch

train_labels = torch.tensor(train_df['label'].values)
val_labels = torch.tensor(val_df['label'].values)
test_labels = torch.tensor(test_df['label'].values)

(619150, 3)
(176900, 3)
(88451, 3)


In [9]:
import torch

encodings_path_tarin = './BERT_encodings/train_encodings.pt'
encodings_path_val = './BERT_encodings/val_encodings.pt'
encodings_path_test = './BERT_encodings/test_encodings.pt'

encoding_dir = './BERT_encodings'
# Create output directory if it doesn't exist
if not os.path.exists(encoding_dir):
    os.makedirs(encoding_dir)
# Save encodings and labels
torch.save({'input_ids': train_encodings['input_ids'],
            'attention_mask': train_encodings['attention_mask'],
            'labels': train_labels}, encodings_path_tarin)

torch.save({'input_ids': val_encodings['input_ids'],
            'attention_mask': val_encodings['attention_mask'],
            'labels': val_labels}, encodings_path_val)

torch.save({'input_ids': test_encodings['input_ids'],
            'attention_mask': test_encodings['attention_mask'],
            'labels': test_labels}, encodings_path_test)


In [10]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Load the saved encodings and labels
train_data = torch.load(encodings_path_tarin)
val_data = torch.load(encodings_path_val)
test_data = torch.load(encodings_path_test)

# Extract tensors from the dictionaries
train_input_ids = train_data['input_ids']
train_attention_mask = train_data['attention_mask']
train_labels = train_data['labels']

val_input_ids = val_data['input_ids']
val_attention_mask = val_data['attention_mask']
val_labels = val_data['labels']

test_input_ids = test_data['input_ids']
test_attention_mask = test_data['attention_mask']
test_labels = test_data['labels']

<ipython-input-10-746cb582f02f>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(encodings_path_tarin)
<ipython-input-10-746cb582f02f>:6: FutureWarni

In [ ]:
# Create TensorDatasets
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
val_dataset = TensorDataset(val_input_ids, val_attention_mask, val_labels)
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels)

In [ ]:
# Define batch sizes
batch_size = 32

# Create DataLoaders
# For the training set, we usually want to shuffle the data
train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size
)

# For validation and test sets, we usually do not shuffle, so we use a SequentialSampler
val_dataloader = DataLoader(
    val_dataset,
    sampler=SequentialSampler(val_dataset),
    batch_size=batch_size
)

test_dataloader = DataLoader(
    test_dataset,
    sampler=SequentialSampler(test_dataset),
    batch_size=batch_size
)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Number of labels for binary classification
num_labels = 2

# Load the pretrained BERT model with a classification head on top
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Setup the optimizer
# AdamW is a commonly used optimizer for fine-tuning BERT
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,               # Suggested starting learning rate for BERT fine-tuning
    eps=1e-8               # AdamW epsilon parameter
)

# Total number of training steps
# Typically: steps_per_epoch = len(train_dataloader)
# total_steps = steps_per_epoch * num_epochs
# For example, if you have 10,000 training steps total and 3 epochs, total_steps = 30000.
steps_per_epoch = len(train_dataloader)
num_epochs = 3
total_steps = steps_per_epoch * num_epochs

# Create a linear warmup and decay learning rate schedule
# Usually we warm up for ~10% of total steps
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Loss Function
# For AutoModelForSequenceClassification, the forward pass returns loss if labels are provided.
# Typically, we don’t need to set the loss function manually, as the model integrates it internally.
# However, if you want explicit reference, you can use:
criterion = torch.nn.CrossEntropyLoss()


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
import numpy as np
from tqdm.auto import tqdm

def compute_accuracy(preds, labels):
    """Compute accuracy given predictions (logits) and true labels."""
    preds = np.argmax(preds, axis=1)
    return (preds == labels).mean()

def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    """Train the model for one epoch."""
    model.train()
    total_loss = 0.0
    total_accuracy = 0.0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        b_input_ids = batch[0].to(device)
        b_attention_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        optimizer.zero_grad()

        # Forward pass (with labels returns loss)
        outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask, labels=b_labels)
        loss = outputs.loss
        logits = outputs.logits

        # Accumulate loss
        total_loss += loss.item()

        # Compute accuracy
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()
        total_accuracy += compute_accuracy(logits, label_ids)

        # Backpropagation
        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(dataloader)
    avg_accuracy = total_accuracy / len(dataloader)
    return avg_loss, avg_accuracy


def evaluate(model, dataloader, device):
    """Evaluate the model on the given dataloader."""
    model.eval()
    total_loss = 0.0
    total_accuracy = 0.0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            b_input_ids = batch[0].to(device)
            b_attention_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask, labels=b_labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.cpu().numpy()
            total_accuracy += compute_accuracy(logits, label_ids)

    avg_loss = total_loss / len(dataloader)
    avg_accuracy = total_accuracy / len(dataloader)
    return avg_loss, avg_accuracy


In [ ]:
epochs = 3
for epoch in range(epochs):
    print(f"\n======== Epoch {epoch+1}/{epochs} ========")

    # Train for one epoch
    train_loss, train_accuracy = train_one_epoch(model, train_dataloader, optimizer, scheduler, device)
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.4f}")

    # Evaluate on validation set
    val_loss, val_accuracy = evaluate(model, val_dataloader, device)
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

print("Training complete.")

import os

output_dir = "./BERT_MODEL/"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the model weights and configuration
model.save_pretrained(output_dir)

# If you're using a pretrained tokenizer, save it as well
# tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")



======== Epoch 1/3 ========


Training:   0%|          | 0/19349 [00:00<?, ?it/s]

Training Loss: 0.4225, Training Accuracy: 0.8043


Evaluating:   0%|          | 0/5529 [00:00<?, ?it/s]

Validation Loss: 0.3962, Validation Accuracy: 0.8211

======== Epoch 2/3 ========


...

Validation Loss: 0.3925, Validation Accuracy: 0.8246

======== Epoch 3/3 ========


Training:   0%|          | 0/19349 [00:00<?, ?it/s]

Training Loss: 0.3466, Training Accuracy: 0.8504


Evaluating:   0%|          | 0/5529 [00:00<?, ?it/s]

Validation Loss: 0.4083, Validation Accuracy: 0.8226
Training complete.
Model saved to ./Trained_saved_model/


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Put the model in evaluation mode
model.eval()

all_preds = []
all_labels = []

# Disable gradient calculations for evaluation
with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids = batch[0].to(device)
        b_attention_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # Forward pass
        outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
        logits = outputs.logits

        # Get predictions
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = b_labels.cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

# Compute metrics
print("Classification Report:")
print(classification_report(all_labels, all_preds, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

# Compute accuracy
accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Test Accuracy: {accuracy:.4f}")

# Show some example predictions from the test set
# Assuming test_df is aligned with test_dataloader (same ordering)
# If not aligned, you may need to ensure you have the same indexing or track the indices.
# Here, we assume test_df is in the same order as when we created test_dataloader.
num_examples_to_show = 5
print("\nSample Predictions:")
for i in range(num_examples_to_show):
    text = test_df['text'].iloc[i]
    true_label = test_df['label'].iloc[i]
    pred_label = all_preds[i]
    print(f"Text: {text[:100]}...")  # print first 100 chars for brevity
    print(f"True Label: {true_label}, Predicted Label: {pred_label}")
    print("-" * 50)


Classification Report:
              precision    recall  f1-score   support

           0     0.8604    0.5645    0.6817     29584
           1     0.8134    0.9540    0.8781     58867

    accuracy                         0.8237     88451
   macro avg     0.8369    0.7592    0.7799     88451
weighted avg     0.8291    0.8237    0.8124     88451

Confusion Matrix:
[[16700 12884]
 [ 2710 56157]]
Test Accuracy: 0.8237

Sample Predictions:
Text: one best purchase ive made recently customizable setting really make stand kitchen gadget ive tried ...
True Label: 0, Predicted Label: 0
--------------------------------------------------
Text: yorkie oral surgery teeth gotten bad specialist sold u enzymatic toothpaste poultry flavor yorkie lo...
True Label: 1, Predicted Label: 1
--------------------------------------------------
Text: 3 yr old son dairy egg allergy hard find good cookie substitute love cant roll use cookie cutter sti...
True Label: 1, Predicted Label: 1
------------------------